# Module 1: Building a Genetic Algorithm from Scratch

## Learning Objectives

By completing this notebook, you will:
1. Understand the core components of a genetic algorithm
2. Implement binary chromosome encoding for feature selection
3. Build selection operators (tournament, roulette wheel, rank-based)
4. Implement crossover operators (single-point, two-point, uniform)
5. Design mutation strategies with adaptive rates
6. Create a complete GA for feature selection without using DEAP
7. Analyze GA behavior through visualization

## Prerequisites

- Module 0 completed (feature selection comparison)
- Understanding of evolutionary concepts
- Basic NumPy and Python OOP

## Estimated Time: 90 minutes

---

In [ ]:
learning_objectives(["Understand the core components of a genetic algorithm", "Implement binary chromosome encoding for feature selection", "Build selection operators (tournament, roulette wheel, rank-based)", "Implement crossover operators (single-point, two-point, uniform)", "Design mutation strategies with adaptive rates", "Create a complete GA for feature selection without using DEAP", "Analyze GA behavior through visualization"])

In [ ]:
section_divider("Setup")

## 1. Setup

Import all necessary libraries and prepare our test dataset.

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from typing import List, Tuple, Callable
import warnings
warnings.filterwarnings('ignore')

# Set random seed
np.random.seed(42)

# Plotting configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Libraries imported successfully!")

In [ ]:
apply_course_theme()
apply_plot_theme()

### 1.1 Load and Prepare Dataset

We'll use the breast cancer dataset as our feature selection testbed.

In [ ]:
# Purpose: Prepare dataset for GA feature selection experiments
# Key Concept: Consistent train/test split for fair comparison

# Load data
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X.columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X.columns,
    index=X_test.index
)

print(f"Dataset: {X_train_scaled.shape[0]} train samples, {X_test_scaled.shape[0]} test samples")
print(f"Features: {X_train_scaled.shape[1]}")
print(f"Class balance: {np.sum(y_train == 0)} / {np.sum(y_train == 1)}")

In [ ]:
section_divider("Individual Representation")

## 2. Individual Representation

### Key Concept: Binary encoding represents feature selection as a chromosome.

Each gene is a binary value:
- **1**: Feature is selected
- **0**: Feature is not selected

Example: `[1, 0, 1, 1, 0]` selects features 0, 2, and 3 from a 5-feature set.

In [ ]:
callout("Binary encoding represents feature selection as a chromosome.", kind="insight")

In [ ]:
# Purpose: Define Individual class to represent a solution
# Key Concept: Encapsulate chromosome and fitness in one object

class Individual:
    """
    Represents a candidate solution for feature selection.
    
    Attributes
    ----------
    chromosome : np.ndarray
        Binary array indicating selected features
    fitness : float or None
        Fitness score (None if not yet evaluated)
    """
    
    def __init__(self, chromosome: np.ndarray):
        """Initialize individual with binary chromosome."""
        self.chromosome = chromosome.astype(int)
        self.fitness = None
    
    @classmethod
    def random(cls, n_features: int):
        """Create random individual."""
        chromosome = np.random.randint(0, 2, size=n_features)
        return cls(chromosome)
    
    def selected_features(self) -> np.ndarray:
        """Get indices of selected features."""
        return np.where(self.chromosome == 1)[0]
    
    def n_selected(self) -> int:
        """Count number of selected features."""
        return np.sum(self.chromosome)
    
    def copy(self):
        """Create deep copy of individual."""
        new_ind = Individual(self.chromosome.copy())
        new_ind.fitness = self.fitness
        return new_ind
    
    def __repr__(self):
        """String representation."""
        return f"Individual(features={self.n_selected()}, fitness={self.fitness:.4f if self.fitness else 'None'})"

# Test Individual class
test_ind = Individual.random(n_features=10)
print(f"Random individual: {test_ind}")
print(f"Chromosome: {test_ind.chromosome}")
print(f"Selected features: {test_ind.selected_features()}")

In [ ]:
section_divider("Fitness Function")

## 3. Fitness Function

### Key Concept: Fitness function evaluates how good a solution is.

For feature selection, we want:
1. **High accuracy**: Good predictive performance
2. **Few features**: Parsimony (avoid overfitting)
3. **Valid solution**: At least one feature selected

In [ ]:
callout("Fitness function evaluates how good a solution is.", kind="insight")

In [ ]:
# Purpose: Define fitness function for feature selection
# Key Concept: Balance accuracy and parsimony with penalty term

def evaluate_fitness(individual: Individual, X: pd.DataFrame, y: np.ndarray, 
                     parsimony_weight: float = 0.01) -> float:
    """
    Evaluate fitness of a feature selection individual.
    
    Parameters
    ----------
    individual : Individual
        Candidate solution to evaluate
    X : DataFrame
        Feature matrix
    y : array
        Target variable
    parsimony_weight : float
        Weight for parsimony penalty (default 0.01)
    
    Returns
    -------
    fitness : float
        Fitness score (higher is better)
    """
    # Check if at least one feature is selected
    if individual.n_selected() == 0:
        return -999.0  # Very bad fitness for empty selection
    
    # Get selected features
    feature_mask = individual.chromosome.astype(bool)
    X_selected = X.iloc[:, feature_mask]
    
    # Evaluate with cross-validation
    model = LogisticRegression(max_iter=1000, random_state=42)
    cv_scores = cross_val_score(
        model, X_selected, y, cv=5, scoring='accuracy'
    )
    accuracy = cv_scores.mean()
    
    # Calculate parsimony penalty
    # Penalty increases with number of features
    parsimony_penalty = parsimony_weight * (individual.n_selected() / len(individual.chromosome))
    
    # Combined fitness (higher is better)
    fitness = accuracy - parsimony_penalty
    
    return fitness

# Test fitness function
test_ind = Individual.random(n_features=X_train_scaled.shape[1])
test_fitness = evaluate_fitness(test_ind, X_train_scaled, y_train)
test_ind.fitness = test_fitness

print(f"Test individual: {test_ind}")
print(f"Selected {test_ind.n_selected()}/{len(test_ind.chromosome)} features")

In [ ]:
section_divider("Population Initialization")

## 4. Population Initialization

### Key Concept: Start with diverse population to explore search space.

We'll create a population with varying numbers of selected features.

In [ ]:
callout("Start with diverse population to explore search space.", kind="insight")

In [ ]:
# Purpose: Initialize diverse population
# Key Concept: Random initialization with evaluation

def initialize_population(pop_size: int, n_features: int, 
                         X: pd.DataFrame, y: np.ndarray) -> List[Individual]:
    """
    Create and evaluate initial population.
    
    Parameters
    ----------
    pop_size : int
        Number of individuals in population
    n_features : int
        Number of features (chromosome length)
    X : DataFrame
        Training features
    y : array
        Training target
    
    Returns
    -------
    population : list of Individual
        Evaluated population
    """
    population = []
    
    for i in range(pop_size):
        # Create random individual
        individual = Individual.random(n_features)
        
        # Ensure at least one feature is selected
        while individual.n_selected() == 0:
            individual = Individual.random(n_features)
        
        # Evaluate fitness
        individual.fitness = evaluate_fitness(individual, X, y)
        
        population.append(individual)
    
    return population

# Create test population
test_pop = initialize_population(
    pop_size=10,
    n_features=X_train_scaled.shape[1],
    X=X_train_scaled,
    y=y_train
)

print(f"Created population of {len(test_pop)} individuals")
print(f"\nSample individuals:")
for i, ind in enumerate(test_pop[:3]):
    print(f"  {i+1}. {ind}")

# Population statistics
fitnesses = [ind.fitness for ind in test_pop]
n_features_selected = [ind.n_selected() for ind in test_pop]
print(f"\nPopulation statistics:")
print(f"  Fitness - Mean: {np.mean(fitnesses):.4f}, Std: {np.std(fitnesses):.4f}")
print(f"  Features - Mean: {np.mean(n_features_selected):.1f}, Range: [{np.min(n_features_selected)}, {np.max(n_features_selected)}]")

In [ ]:
section_divider("Selection Operators")

## 5. Selection Operators

### Key Concept: Selection determines which individuals reproduce.

We'll implement three common selection methods:
1. **Tournament Selection**: Pick best from random subset
2. **Roulette Wheel**: Probability proportional to fitness
3. **Rank Selection**: Probability based on rank, not raw fitness

### 5.1 Tournament Selection

Randomly select k individuals and return the best one. Simple and effective.

In [ ]:
# Purpose: Implement tournament selection
# Key Concept: Local competition, tunable selection pressure

def tournament_selection(population: List[Individual], 
                        tournament_size: int = 3) -> Individual:
    """
    Select individual via tournament selection.
    
    Parameters
    ----------
    population : list of Individual
        Current population
    tournament_size : int
        Number of individuals in tournament (default 3)
    
    Returns
    -------
    winner : Individual
        Selected individual (copy)
    """
    # Randomly select tournament_size individuals
    tournament = np.random.choice(population, size=tournament_size, replace=False)
    
    # Return best individual (by fitness)
    winner = max(tournament, key=lambda ind: ind.fitness)
    
    return winner.copy()

# Test tournament selection
print("Tournament Selection Test:")
for i in range(3):
    selected = tournament_selection(test_pop, tournament_size=3)
    print(f"  Selection {i+1}: {selected}")

### 5.2 Roulette Wheel Selection

Selection probability proportional to fitness. Better individuals more likely to be selected.

In [ ]:
# Purpose: Implement fitness-proportionate selection
# Key Concept: Probability = fitness / sum(all_fitness)

def roulette_wheel_selection(population: List[Individual]) -> Individual:
    """
    Select individual via roulette wheel (fitness-proportionate).
    
    Parameters
    ----------
    population : list of Individual
        Current population
    
    Returns
    -------
    selected : Individual
        Selected individual (copy)
    """
    # Get fitness values
    fitnesses = np.array([ind.fitness for ind in population])
    
    # Handle negative fitness by shifting
    if np.min(fitnesses) < 0:
        fitnesses = fitnesses - np.min(fitnesses) + 1e-6
    
    # Calculate selection probabilities
    total_fitness = np.sum(fitnesses)
    probabilities = fitnesses / total_fitness
    
    # Select individual
    selected_idx = np.random.choice(len(population), p=probabilities)
    
    return population[selected_idx].copy()

# Test roulette wheel selection
print("Roulette Wheel Selection Test:")
for i in range(3):
    selected = roulette_wheel_selection(test_pop)
    print(f"  Selection {i+1}: {selected}")

### 5.3 Rank Selection

Selection based on rank rather than raw fitness. Reduces selection pressure when fitness variance is high.

In [ ]:
# Purpose: Implement rank-based selection
# Key Concept: Probability based on sorted position, not raw fitness

def rank_selection(population: List[Individual]) -> Individual:
    """
    Select individual via rank selection.
    
    Parameters
    ----------
    population : list of Individual
        Current population
    
    Returns
    -------
    selected : Individual
        Selected individual (copy)
    """
    # Sort population by fitness
    sorted_pop = sorted(population, key=lambda ind: ind.fitness)
    
    # Assign ranks (1 to N)
    ranks = np.arange(1, len(population) + 1)
    
    # Calculate selection probabilities (proportional to rank)
    probabilities = ranks / np.sum(ranks)
    
    # Select individual
    selected_idx = np.random.choice(len(sorted_pop), p=probabilities)
    
    return sorted_pop[selected_idx].copy()

# Test rank selection
print("Rank Selection Test:")
for i in range(3):
    selected = rank_selection(test_pop)
    print(f"  Selection {i+1}: {selected}")

In [ ]:
section_divider("Crossover Operators")

## 6. Crossover Operators

### Key Concept: Crossover combines genetic material from two parents.

We'll implement:
1. **Single-Point Crossover**: Split at one point
2. **Two-Point Crossover**: Split at two points
3. **Uniform Crossover**: Randomly inherit each gene

### 6.1 Single-Point Crossover

Choose random split point, exchange tails.

In [ ]:
# Purpose: Implement single-point crossover
# Key Concept: Split chromosome at one point, exchange tails

def single_point_crossover(parent1: Individual, parent2: Individual) -> Tuple[Individual, Individual]:
    """
    Perform single-point crossover on two parents.
    
    Parameters
    ----------
    parent1, parent2 : Individual
        Parent individuals
    
    Returns
    -------
    child1, child2 : Individual
        Offspring individuals
    """
    # Choose random crossover point
    point = np.random.randint(1, len(parent1.chromosome))
    
    # Create offspring
    child1_chrom = np.concatenate([parent1.chromosome[:point], parent2.chromosome[point:]])
    child2_chrom = np.concatenate([parent2.chromosome[:point], parent1.chromosome[point:]])
    
    child1 = Individual(child1_chrom)
    child2 = Individual(child2_chrom)
    
    return child1, child2

# Test single-point crossover
p1 = Individual(np.array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0]))
p2 = Individual(np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1]))
c1, c2 = single_point_crossover(p1, p2)

print("Single-Point Crossover Test:")
print(f"  Parent 1: {p1.chromosome}")
print(f"  Parent 2: {p2.chromosome}")
print(f"  Child 1:  {c1.chromosome}")
print(f"  Child 2:  {c2.chromosome}")

### 6.2 Two-Point Crossover

Choose two split points, exchange middle segment.

In [ ]:
# Purpose: Implement two-point crossover
# Key Concept: Exchange middle segment between two points

def two_point_crossover(parent1: Individual, parent2: Individual) -> Tuple[Individual, Individual]:
    """
    Perform two-point crossover on two parents.
    
    Parameters
    ----------
    parent1, parent2 : Individual
        Parent individuals
    
    Returns
    -------
    child1, child2 : Individual
        Offspring individuals
    """
    # Choose two random crossover points
    points = sorted(np.random.choice(range(1, len(parent1.chromosome)), size=2, replace=False))
    point1, point2 = points
    
    # Create offspring by exchanging middle segment
    child1_chrom = np.concatenate([
        parent1.chromosome[:point1],
        parent2.chromosome[point1:point2],
        parent1.chromosome[point2:]
    ])
    
    child2_chrom = np.concatenate([
        parent2.chromosome[:point1],
        parent1.chromosome[point1:point2],
        parent2.chromosome[point2:]
    ])
    
    child1 = Individual(child1_chrom)
    child2 = Individual(child2_chrom)
    
    return child1, child2

# Test two-point crossover
c1, c2 = two_point_crossover(p1, p2)

print("Two-Point Crossover Test:")
print(f"  Parent 1: {p1.chromosome}")
print(f"  Parent 2: {p2.chromosome}")
print(f"  Child 1:  {c1.chromosome}")
print(f"  Child 2:  {c2.chromosome}")

### 6.3 Uniform Crossover

Each gene is independently inherited from either parent with 50% probability.

In [ ]:
# Purpose: Implement uniform crossover
# Key Concept: Each gene independently inherited from random parent

def uniform_crossover(parent1: Individual, parent2: Individual, 
                     prob: float = 0.5) -> Tuple[Individual, Individual]:
    """
    Perform uniform crossover on two parents.
    
    Parameters
    ----------
    parent1, parent2 : Individual
        Parent individuals
    prob : float
        Probability of inheriting from parent1 (default 0.5)
    
    Returns
    -------
    child1, child2 : Individual
        Offspring individuals
    """
    # Create random mask
    mask = np.random.random(len(parent1.chromosome)) < prob
    
    # Create offspring
    child1_chrom = np.where(mask, parent1.chromosome, parent2.chromosome)
    child2_chrom = np.where(mask, parent2.chromosome, parent1.chromosome)
    
    child1 = Individual(child1_chrom)
    child2 = Individual(child2_chrom)
    
    return child1, child2

# Test uniform crossover
c1, c2 = uniform_crossover(p1, p2)

print("Uniform Crossover Test:")
print(f"  Parent 1: {p1.chromosome}")
print(f"  Parent 2: {p2.chromosome}")
print(f"  Child 1:  {c1.chromosome}")
print(f"  Child 2:  {c2.chromosome}")

In [ ]:
section_divider("Mutation Operator")

## 7. Mutation Operator

### Key Concept: Mutation introduces random variation to maintain diversity.

For binary encoding, mutation flips bits with a certain probability.

In [ ]:
# Purpose: Implement bit-flip mutation
# Key Concept: Flip each bit with probability mutation_rate

def mutate(individual: Individual, mutation_rate: float = 0.05) -> Individual:
    """
    Perform bit-flip mutation on individual.
    
    Parameters
    ----------
    individual : Individual
        Individual to mutate (modified in place)
    mutation_rate : float
        Probability of flipping each bit (default 0.05)
    
    Returns
    -------
    individual : Individual
        Mutated individual
    """
    # Create mutation mask
    mutation_mask = np.random.random(len(individual.chromosome)) < mutation_rate
    
    # Flip bits where mask is True
    individual.chromosome[mutation_mask] = 1 - individual.chromosome[mutation_mask]
    
    # Ensure at least one feature is selected
    if individual.n_selected() == 0:
        # Randomly select one feature
        random_idx = np.random.randint(0, len(individual.chromosome))
        individual.chromosome[random_idx] = 1
    
    # Reset fitness (needs re-evaluation)
    individual.fitness = None
    
    return individual

# Test mutation
test_ind = Individual(np.array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0]))
print("Mutation Test:")
print(f"  Original: {test_ind.chromosome}")

for i in range(3):
    mutated = test_ind.copy()
    mutate(mutated, mutation_rate=0.2)  # High rate for demonstration
    print(f"  Mutated {i+1}: {mutated.chromosome}")

In [ ]:
section_divider("Complete Genetic Algorithm")

## 8. Complete Genetic Algorithm

### Key Concept: Combine all components into a complete GA loop.

The standard GA cycle:
1. **Initialize** population
2. **Evaluate** fitness
3. **Select** parents
4. **Crossover** to create offspring
5. **Mutate** offspring
6. **Replace** population
7. **Repeat** until termination

In [ ]:
# Purpose: Implement complete genetic algorithm
# Key Concept: Generational GA with elitism

class GeneticAlgorithm:
    """
    Genetic Algorithm for feature selection.
    """
    
    def __init__(self, 
                 pop_size: int = 50,
                 n_generations: int = 50,
                 crossover_rate: float = 0.7,
                 mutation_rate: float = 0.05,
                 tournament_size: int = 3,
                 elitism: int = 2,
                 selection_method: str = 'tournament',
                 crossover_method: str = 'two_point'):
        """
        Initialize GA parameters.
        
        Parameters
        ----------
        pop_size : int
            Population size
        n_generations : int
            Number of generations
        crossover_rate : float
            Probability of crossover
        mutation_rate : float
            Probability of mutation per bit
        tournament_size : int
            Tournament size for selection
        elitism : int
            Number of best individuals to preserve
        selection_method : str
            'tournament', 'roulette', or 'rank'
        crossover_method : str
            'single_point', 'two_point', or 'uniform'
        """
        self.pop_size = pop_size
        self.n_generations = n_generations
        self.crossover_rate = crossover_rate
        self.mutation_rate = mutation_rate
        self.tournament_size = tournament_size
        self.elitism = elitism
        self.selection_method = selection_method
        self.crossover_method = crossover_method
        
        # Statistics
        self.best_individual = None
        self.best_fitness_history = []
        self.avg_fitness_history = []
        self.diversity_history = []
    
    def select_parent(self, population: List[Individual]) -> Individual:
        """Select parent using configured method."""
        if self.selection_method == 'tournament':
            return tournament_selection(population, self.tournament_size)
        elif self.selection_method == 'roulette':
            return roulette_wheel_selection(population)
        elif self.selection_method == 'rank':
            return rank_selection(population)
        else:
            raise ValueError(f"Unknown selection method: {self.selection_method}")
    
    def crossover(self, parent1: Individual, parent2: Individual) -> Tuple[Individual, Individual]:
        """Perform crossover using configured method."""
        if self.crossover_method == 'single_point':
            return single_point_crossover(parent1, parent2)
        elif self.crossover_method == 'two_point':
            return two_point_crossover(parent1, parent2)
        elif self.crossover_method == 'uniform':
            return uniform_crossover(parent1, parent2)
        else:
            raise ValueError(f"Unknown crossover method: {self.crossover_method}")
    
    def run(self, X: pd.DataFrame, y: np.ndarray, verbose: bool = True) -> Individual:
        """
        Run genetic algorithm.
        
        Parameters
        ----------
        X : DataFrame
            Training features
        y : array
            Training target
        verbose : bool
            Print progress
        
        Returns
        -------
        best_individual : Individual
            Best solution found
        """
        n_features = X.shape[1]
        
        # Initialize population
        population = initialize_population(self.pop_size, n_features, X, y)
        
        # Track best individual
        self.best_individual = max(population, key=lambda ind: ind.fitness)
        
        if verbose:
            print(f"Generation 0: Best fitness = {self.best_individual.fitness:.4f}, "
                  f"Features = {self.best_individual.n_selected()}")
        
        # Evolution loop
        for gen in range(self.n_generations):
            # Create new population
            new_population = []
            
            # Elitism: preserve best individuals
            if self.elitism > 0:
                elite = sorted(population, key=lambda ind: ind.fitness, reverse=True)[:self.elitism]
                new_population.extend([ind.copy() for ind in elite])
            
            # Generate offspring
            while len(new_population) < self.pop_size:
                # Select parents
                parent1 = self.select_parent(population)
                parent2 = self.select_parent(population)
                
                # Crossover
                if np.random.random() < self.crossover_rate:
                    child1, child2 = self.crossover(parent1, parent2)
                else:
                    child1, child2 = parent1.copy(), parent2.copy()
                
                # Mutation
                mutate(child1, self.mutation_rate)
                mutate(child2, self.mutation_rate)
                
                # Evaluate offspring
                child1.fitness = evaluate_fitness(child1, X, y)
                child2.fitness = evaluate_fitness(child2, X, y)
                
                # Add to new population
                new_population.append(child1)
                if len(new_population) < self.pop_size:
                    new_population.append(child2)
            
            # Replace population
            population = new_population[:self.pop_size]
            
            # Update best individual
            gen_best = max(population, key=lambda ind: ind.fitness)
            if gen_best.fitness > self.best_individual.fitness:
                self.best_individual = gen_best.copy()
            
            # Track statistics
            fitnesses = [ind.fitness for ind in population]
            self.best_fitness_history.append(self.best_individual.fitness)
            self.avg_fitness_history.append(np.mean(fitnesses))
            
            # Calculate diversity (std of number of features selected)
            n_features_selected = [ind.n_selected() for ind in population]
            self.diversity_history.append(np.std(n_features_selected))
            
            # Print progress
            if verbose and (gen + 1) % 10 == 0:
                print(f"Generation {gen + 1}: Best fitness = {self.best_individual.fitness:.4f}, "
                      f"Avg fitness = {self.avg_fitness_history[-1]:.4f}, "
                      f"Features = {self.best_individual.n_selected()}")
        
        return self.best_individual

print("GeneticAlgorithm class defined!")

### 8.1 Run the Genetic Algorithm

Let's run our custom GA on the breast cancer dataset.

In [ ]:
# Purpose: Run custom GA for feature selection
# Key Concept: Evolve population to find optimal feature subset

# Create GA instance
ga = GeneticAlgorithm(
    pop_size=50,
    n_generations=50,
    crossover_rate=0.7,
    mutation_rate=0.05,
    tournament_size=3,
    elitism=2,
    selection_method='tournament',
    crossover_method='two_point'
)

# Run GA
print("Running Genetic Algorithm...\n")
best_solution = ga.run(X_train_scaled, y_train, verbose=True)

print("\n" + "="*70)
print("GA Evolution Complete!")
print(f"\nBest solution found:")
print(f"  Fitness: {best_solution.fitness:.4f}")
print(f"  Features selected: {best_solution.n_selected()}/{len(best_solution.chromosome)}")
print(f"  Selected feature indices: {best_solution.selected_features()}")

### 8.2 Visualize Evolution

Understand how the GA converged by plotting fitness and diversity over time.

In [ ]:
# Purpose: Visualize GA evolution dynamics
# Key Concept: Monitor convergence and diversity loss

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

generations = range(1, len(ga.best_fitness_history) + 1)

# Plot 1: Fitness evolution
axes[0, 0].plot(generations, ga.best_fitness_history, 'r-', linewidth=2, label='Best')
axes[0, 0].plot(generations, ga.avg_fitness_history, 'b-', linewidth=2, label='Average')
axes[0, 0].set_xlabel('Generation', fontsize=12)
axes[0, 0].set_ylabel('Fitness', fontsize=12)
axes[0, 0].set_title('Fitness Evolution', fontsize=14)
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Plot 2: Diversity evolution
axes[0, 1].plot(generations, ga.diversity_history, 'g-', linewidth=2)
axes[0, 1].set_xlabel('Generation', fontsize=12)
axes[0, 1].set_ylabel('Diversity (Std of #Features)', fontsize=12)
axes[0, 1].set_title('Population Diversity', fontsize=14)
axes[0, 1].grid(alpha=0.3)

# Plot 3: Convergence rate
improvement = np.diff(ga.best_fitness_history)
axes[1, 0].plot(range(2, len(ga.best_fitness_history) + 1), improvement, 'purple', linewidth=2)
axes[1, 0].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[1, 0].set_xlabel('Generation', fontsize=12)
axes[1, 0].set_ylabel('Fitness Improvement', fontsize=12)
axes[1, 0].set_title('Convergence Rate', fontsize=14)
axes[1, 0].grid(alpha=0.3)

# Plot 4: Selected features visualization
feature_names = X_train_scaled.columns[best_solution.selected_features()]
axes[1, 1].barh(range(len(feature_names)), [1]*len(feature_names), color='steelblue')
axes[1, 1].set_yticks(range(len(feature_names)))
axes[1, 1].set_yticklabels(feature_names, fontsize=8)
axes[1, 1].set_xlabel('Selected', fontsize=12)
axes[1, 1].set_title(f'Selected Features ({len(feature_names)})', fontsize=14)
axes[1, 1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print(f"  - Best fitness improved by {ga.best_fitness_history[-1] - ga.best_fitness_history[0]:.4f}")
print(f"  - Diversity decreased from {ga.diversity_history[0]:.2f} to {ga.diversity_history[-1]:.2f}")
print(f"  - Convergence mostly complete by generation {np.argmax(np.array(improvement) < 0.001) + 1}")

### 8.3 Evaluate on Test Set

Assess generalization performance of the selected features.

In [ ]:
# Purpose: Evaluate GA-selected features on held-out test set
# Key Concept: Test set performance indicates generalization

# Get selected features
selected_features = best_solution.selected_features()
feature_names = X_train_scaled.columns[selected_features]

# Train model on selected features
X_train_selected = X_train_scaled.iloc[:, selected_features]
X_test_selected = X_test_scaled.iloc[:, selected_features]

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_selected, y_train)

# Evaluate
train_score = model.score(X_train_selected, y_train)
test_score = model.score(X_test_selected, y_test)
cv_score = cross_val_score(model, X_train_selected, y_train, cv=5, scoring='accuracy').mean()

# Baseline: all features
model_baseline = LogisticRegression(max_iter=1000, random_state=42)
model_baseline.fit(X_train_scaled, y_train)
baseline_train = model_baseline.score(X_train_scaled, y_train)
baseline_test = model_baseline.score(X_test_scaled, y_test)

print("Performance Comparison")
print("="*70)
print(f"\nGA-Selected Features ({len(selected_features)} features):")
print(f"  Train Accuracy: {train_score:.4f}")
print(f"  CV Accuracy:    {cv_score:.4f}")
print(f"  Test Accuracy:  {test_score:.4f}")

print(f"\nBaseline (All {X_train_scaled.shape[1]} features):")
print(f"  Train Accuracy: {baseline_train:.4f}")
print(f"  Test Accuracy:  {baseline_test:.4f}")

print(f"\nImprovement:")
print(f"  Test accuracy change: {(test_score - baseline_test):.4f}")
print(f"  Feature reduction: {X_train_scaled.shape[1] - len(selected_features)} features removed")
print(f"  Percentage reduction: {100 * (1 - len(selected_features)/X_train_scaled.shape[1]):.1f}%")

In [ ]:
section_divider("Exercises")

## 9. Exercises

### Exercise 9.1: Compare Selection Methods

**Task**: Run the GA with all three selection methods (tournament, roulette, rank) and compare their convergence speed and final performance.

**Expected Output**: A plot showing fitness evolution for all three methods.

**Hints**:
<details>
<summary>Hint 1</summary>
Create three GA instances with different selection_method parameters.
</details>

<details>
<summary>Hint 2</summary>
Store the best_fitness_history from each run and plot them on the same axes.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

# TODO: Create three GA instances with different selection methods
# TODO: Run each GA
# TODO: Plot fitness evolution for comparison

In [ ]:
# AUTO-GRADED TEST - Do not modify
# ----------------------------------
def test_exercise_91():
    print("✓ Exercise 9.1: Complete the implementation and visual comparison")

test_exercise_91()

### Exercise 9.2: Adaptive Mutation Rate

**Task**: Implement an adaptive mutation rate that starts high and decreases over generations. Test if this improves performance.

**Expected Output**: Modified GA with adaptive mutation and comparison results.

**Hints**:
<details>
<summary>Hint 1</summary>
Use formula: mutation_rate = initial_rate * (1 - generation / max_generations)
</details>

<details>
<summary>Hint 2</summary>
Modify the GeneticAlgorithm.run() method to update mutation_rate each generation.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

class AdaptiveGA(GeneticAlgorithm):
    """GA with adaptive mutation rate."""
    
    def __init__(self, initial_mutation_rate=0.2, **kwargs):
        super().__init__(**kwargs)
        self.initial_mutation_rate = initial_mutation_rate
    
    # TODO: Override run() method to implement adaptive mutation

In [ ]:
# AUTO-GRADED TEST - Do not modify
# ----------------------------------
def test_exercise_92():
    assert 'AdaptiveGA' in dir(), "Define AdaptiveGA class"
    print("✓ Exercise 9.2: AdaptiveGA class defined")

test_exercise_92()

### Exercise 9.3: Constraint Handling

**Task**: Modify the GA to ensure solutions have exactly k features (e.g., k=10). Implement a repair operator that adds/removes features to meet the constraint.

**Expected Output**: GA that always produces solutions with exactly k features.

**Hints**:
<details>
<summary>Hint 1</summary>
Create a repair() function that randomly adds features if n_selected < k, or removes if n_selected > k.
</details>

<details>
<summary>Hint 2</summary>
Call repair() after crossover and mutation in the GA loop.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

def repair_to_k_features(individual: Individual, k: int) -> Individual:
    """
    Repair individual to have exactly k features.
    
    Parameters
    ----------
    individual : Individual
        Individual to repair
    k : int
        Target number of features
    
    Returns
    -------
    individual : Individual
        Repaired individual
    """
    # TODO: Implement repair logic
    pass

# Test repair function
# test_ind = Individual.random(20)
# print(f"Before repair: {test_ind.n_selected()} features")
# repair_to_k_features(test_ind, k=10)
# print(f"After repair: {test_ind.n_selected()} features")

In [ ]:
# AUTO-GRADED TEST - Do not modify
# ----------------------------------
def test_exercise_93():
    assert 'repair_to_k_features' in dir(), "Define repair_to_k_features function"
    
    # Test repair
    test_ind = Individual(np.ones(20, dtype=int))  # All features selected
    try:
        repair_to_k_features(test_ind, k=10)
        assert test_ind.n_selected() == 10, f"Should have 10 features, got {test_ind.n_selected()}"
        print("✓ Exercise 9.3 passed!")
    except:
        print("Function exists but needs correct implementation")

# Uncomment to test
# test_exercise_93()

### Exercise 9.4: Multi-Objective Fitness

**Task**: Extend the fitness function to explicitly balance three objectives: accuracy, number of features, and feature correlation (redundancy). Implement weighted sum approach.

**Expected Output**: New fitness function and comparison with standard fitness.

**Hints**:
<details>
<summary>Hint 1</summary>
Calculate mean pairwise correlation among selected features.
</details>

<details>
<summary>Hint 2</summary>
Use weights: w1*accuracy - w2*(n_features/total) - w3*mean_correlation
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

def multi_objective_fitness(individual: Individual, X: pd.DataFrame, y: np.ndarray,
                           w_accuracy: float = 1.0,
                           w_parsimony: float = 0.3,
                           w_redundancy: float = 0.2) -> float:
    """
    Multi-objective fitness function.
    
    Balances:
    1. Accuracy (maximize)
    2. Parsimony - fewer features (maximize)
    3. Low redundancy - less correlation (maximize)
    """
    # TODO: Implement multi-objective fitness
    pass

In [ ]:
# AUTO-GRADED TEST - Do not modify
# ----------------------------------
def test_exercise_94():
    assert 'multi_objective_fitness' in dir(), "Define multi_objective_fitness function"
    print("✓ Exercise 9.4: Function defined")
    print("Test your implementation by comparing results with standard fitness")

test_exercise_94()

### Exercise 9.5: Early Stopping

**Task**: Implement early stopping that terminates the GA if the best fitness hasn't improved for N generations (e.g., N=10).

**Expected Output**: Modified GA that stops early when converged.

**Hints**:
<details>
<summary>Hint 1</summary>
Track the generation number when best fitness last improved.
</details>

<details>
<summary>Hint 2</summary>
Break the evolution loop if (current_gen - last_improvement_gen) > patience.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

# TODO: Modify GeneticAlgorithm.run() to implement early stopping
# Or create a new subclass with early stopping

In [ ]:
section_divider("Solutions")

## 10. Solutions

Solutions to exercises are provided below. Try solving them yourself first!

### Solution 9.3: Constraint Handling

In [ ]:
# Solution for Exercise 9.3
def repair_to_k_features(individual: Individual, k: int) -> Individual:
    """
    Repair individual to have exactly k features.
    """
    current_n = individual.n_selected()
    
    if current_n < k:
        # Add features
        unselected = np.where(individual.chromosome == 0)[0]
        to_add = np.random.choice(unselected, size=k - current_n, replace=False)
        individual.chromosome[to_add] = 1
    
    elif current_n > k:
        # Remove features
        selected = np.where(individual.chromosome == 1)[0]
        to_remove = np.random.choice(selected, size=current_n - k, replace=False)
        individual.chromosome[to_remove] = 0
    
    # Reset fitness
    individual.fitness = None
    
    return individual

# Test
test_ind = Individual(np.ones(20, dtype=int))
print(f"Before: {test_ind.n_selected()} features")
repair_to_k_features(test_ind, k=10)
print(f"After: {test_ind.n_selected()} features")

In [ ]:
section_divider("Summary")

## 11. Summary

### Key Takeaways

1. **Binary Encoding**: Natural representation for feature selection (1=selected, 0=not selected)
2. **Fitness Function**: Balance accuracy and parsimony to avoid overfitting
3. **Selection Methods**: Tournament (simple), Roulette (fitness-proportionate), Rank (robust)
4. **Crossover**: Single-point (simple), Two-point (better), Uniform (most diverse)
5. **Mutation**: Essential for maintaining diversity and escaping local optima
6. **Elitism**: Preserving best solutions accelerates convergence
7. **Population Dynamics**: Monitor diversity to detect premature convergence

### GA Design Principles

1. **Balance exploration and exploitation**: High diversity early, convergence later
2. **Avoid premature convergence**: Use adequate mutation and population size
3. **Fitness landscape matters**: Complex landscapes need more diversity
4. **No free lunch**: No single operator configuration works best for all problems
5. **Constraint handling**: Repair operators can enforce hard constraints

### What's Next

In **Module 2**, we'll explore advanced fitness functions:
- Time series-specific fitness
- Multi-objective optimization (Pareto fronts)
- Overfitting prevention strategies
- Domain knowledge integration
- Fitness approximation for speed

### Additional Resources

- **"Introduction to Evolutionary Computing"** by Eiben & Smith (textbook)
- **"Genetic Algorithms in Search, Optimization and Machine Learning"** by Goldberg (classic)
- **PyGAD library**: https://pygad.readthedocs.io/
- **DEAP tutorials**: https://deap.readthedocs.io/en/master/tutorials/

In [ ]:
key_takeaways(["Natural representation for feature selection (1=selected, 0=not selected)", "Balance accuracy and parsimony to avoid overfitting", "Tournament (simple), Roulette (fitness-proportionate), Rank (robust)", "Single-point (simple), Two-point (better), Uniform (most diverse)", "Essential for maintaining diversity and escaping local optima", "Preserving best solutions accelerates convergence", "Monitor diversity to detect premature convergence"])